## Metadata-Routed CCV Comparison

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

if Path.cwd().name.lower() == "visualization":
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

RESULT_ROOT = REPO_ROOT / "results" / "ablation"
FIGURE_DIR = REPO_ROOT / "Visualization" / "Graph"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def load_cache_csv(cache_dir: Path, file_name: str) -> pd.DataFrame:
    path = cache_dir / file_name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

MOE_BASELINE_CACHE = RESULT_ROOT / "moe_baseline"
MOE_SMALL_DATA_CACHE = RESULT_ROOT / "moe_small_data"
MOE_EARLY_CYCLE_CACHE = RESULT_ROOT / "moe_early_cycle" / "cycle_effect"

moe_baseline_overall_metric_df = load_cache_csv(MOE_BASELINE_CACHE, "overall_metric_df.csv")
moe_baseline_dataset_metric_df = load_cache_csv(MOE_BASELINE_CACHE, "dataset_metric_df.csv")
moe_baseline_overall_summary_df = load_cache_csv(MOE_BASELINE_CACHE, "overall_summary_df.csv")
moe_baseline_dataset_summary_df = load_cache_csv(MOE_BASELINE_CACHE, "dataset_summary_df.csv")

moe_small_data_overall_metric_df = load_cache_csv(MOE_SMALL_DATA_CACHE, "overall_metric_df.csv")
moe_small_data_overall_summary_df = load_cache_csv(MOE_SMALL_DATA_CACHE, "overall_summary_df.csv")

moe_early_cycle_overall_summary_df = load_cache_csv(MOE_EARLY_CYCLE_CACHE, "overall_summary_df.csv")


In [ ]:
major_df = moe_baseline_overall_summary_df.loc[moe_baseline_overall_summary_df["plot_group"].eq("major")].copy()
if major_df.empty:
    print("No major benchmark rows found in MoE baseline cache.")
else:
    plt.rcParams.update({
        "text.usetex": False,
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    })

    split_order = ["total split", "per-dataset split", "fine-group split"]
    split_label_map = {
        "total split": "Total split",
        "per-dataset split": "Per-dataset split",
        "fine-group split": "Fine-group Split",
    }

    major_specs = [("CCV-basic", "abs"), ("CCV-meta-MoE", "abs"), ("CCV-meta-MoLE", "abs")]
    label_map = {
        ("CCV-basic", "abs"): "CCV-basic",
        ("CCV-meta-MoE", "abs"): "CCV-meta-MoE",
        ("CCV-meta-MoLE", "abs"): "CCV-meta-MoLE",
    }
    palette = {
        ("CCV-basic", "abs"): "#f89191",
        ("CCV-meta-MoE", "abs"): "#7cb5ec",
        ("CCV-meta-MoLE", "abs"): "#90c987",
    }
    scatter_style = {
        ("CCV-basic", "abs"): dict(marker="D"),
        ("CCV-meta-MoE", "abs"): dict(marker="o"),
        ("CCV-meta-MoLE", "abs"): dict(marker="s"),
    }

    split_min_mape = {}
    for split_name in split_order:
        vals = []
        for spec_key in major_specs:
            model_name, input_mode = spec_key
            row = major_df.loc[
                (major_df["model"] == model_name)
                & (major_df["input_mode"] == input_mode)
                & (major_df["split_protocol"] == split_name)
            ]
            if not row.empty:
                value = float(row["mape"].iloc[0])
                if np.isfinite(value):
                    vals.append(value)
        split_min_mape[split_name] = min(vals) if vals else np.nan

    fig, ax = plt.subplots(figsize=(6.8, 4.2), dpi=300)
    x = np.arange(len(split_order), dtype=float)
    width = 0.24

    for idx, spec_key in enumerate(major_specs):
        model_name, input_mode = spec_key
        sub = (
            major_df.loc[
                major_df["model"].eq(model_name)
                & major_df["input_mode"].eq(input_mode)
            ]
            .set_index("split_protocol")
            .reindex(split_order)
            .reset_index()
        )
        xpos = x + (idx - 1) * width
        mean_values = sub["mape"].to_numpy(dtype=float)
        std_values = sub["mape_std"].fillna(0.0).to_numpy(dtype=float)
        bars = ax.bar(xpos, mean_values, width=width, color=palette[spec_key], edgecolor="none", linewidth=0.0, alpha=0.82, zorder=2, label=label_map[spec_key])
        ax.errorbar(xpos, mean_values, yerr=std_values, fmt="none", capsize=3, elinewidth=1.0, capthick=1.0, ecolor="#000000", alpha=1.0, zorder=5)
        metric_sub = moe_baseline_overall_metric_df.loc[
            moe_baseline_overall_metric_df["plot_group"].eq("major")
            & moe_baseline_overall_metric_df["model"].eq(model_name)
            & moe_baseline_overall_metric_df["input_mode"].eq(input_mode)
        ].copy()
        rng = np.random.default_rng(900 + idx)
        for split_idx, split_name in enumerate(split_order):
            rep_sub = metric_sub.loc[metric_sub["split_protocol"].eq(split_name)].copy()
            if rep_sub.empty:
                continue
            jitter = rng.normal(0.0, width * 0.10, size=len(rep_sub))
            scatter_x = np.full(len(rep_sub), xpos[split_idx], dtype=float) + jitter
            scatter_y = rep_sub["mape"].to_numpy(dtype=float)
            ax.scatter(scatter_x, scatter_y, s=50, facecolor=palette[spec_key], edgecolor="white", linewidth=0.5, zorder=4, **scatter_style[spec_key])
        for bar, mean_v, std_v, split_name in zip(bars, mean_values, std_values, split_order):
            if np.isfinite(mean_v):
                is_min = np.isclose(mean_v, split_min_mape.get(split_name, np.nan)) if np.isfinite(split_min_mape.get(split_name, np.nan)) else False
                ax.text(bar.get_x() + bar.get_width() / 2.0, mean_v + std_v + 0.35, f"{mean_v:.1f}", ha="center", va="bottom", fontsize=10, color="#000000", fontweight="bold" if is_min else "normal", zorder=5)
    ax.set_xticks(x)
    ax.set_xticklabels([split_label_map[s] for s in split_order], fontsize=12)
    ax.set_ylabel("MAPE %", fontsize=12)
    ymax = float(np.nanmax(major_df["mape"].to_numpy(dtype=float))) if not major_df.empty else 30.0
    ax.set_ylim(0, max(12.0, ymax + 5.0))
    ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.22, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.9)
        spine.set_color("#000000")
    ax.tick_params(axis="y", labelsize=10)
    ax.legend(frameon=False, fontsize=11, loc="upper right")
    fig.tight_layout()
    #plt.savefig(FIGURE_DIR / "S13-1.tiff", format="tiff", dpi=500, bbox_inches="tight")
    plt.show()


In [ ]:
major_df = moe_small_data_overall_summary_df.loc[moe_small_data_overall_summary_df["plot_group"].eq("major")].copy()
if major_df.empty:
    print("No major small-data rows found in MoE small-data cache.")
else:
    training_mode_order = ["total split small data", "per-dataset small data", "fine-group small data"]
    available_modes = [name for name in training_mode_order if name in major_df["training_mode"].astype(str).unique().tolist()]
    fraction_order = sorted(major_df["train_fraction_total"].dropna().astype(float).unique().tolist(), reverse=True)
    major_specs = [("CCV-basic", "abs"), ("CCV-meta-MoE", "abs"), ("CCV-meta-MoLE", "abs")]
    label_map = {
        ("CCV-basic", "abs"): "CCV-basic",
        ("CCV-meta-MoE", "abs"): "CCV-meta-MoE",
        ("CCV-meta-MoLE", "abs"): "CCV-meta-MoLE",
    }
    palette = {
        ("CCV-basic", "abs"): "#eb8286",
        ("CCV-meta-MoE", "abs"): "#7fb2d5",
        ("CCV-meta-MoLE", "abs"): "#8dd2c5",
    }
    darker_palette = {
        ("CCV-basic", "abs"): "#c95a5a",
        ("CCV-meta-MoE", "abs"): "#4d8aad",
        ("CCV-meta-MoLE", "abs"): "#5baa9a",
    }
    marker_map = {
        ("CCV-basic", "abs"): "D",
        ("CCV-meta-MoE", "abs"): "o",
        ("CCV-meta-MoLE", "abs"): "s",
    }
    metric_major_df = moe_small_data_overall_metric_df.loc[moe_small_data_overall_metric_df["plot_group"].eq("major")].copy()
    plot_data = []
    for spec_key in major_specs:
        model_name, input_mode = spec_key
        sub_metric = metric_major_df.loc[(metric_major_df["model"] == model_name) & (metric_major_df["input_mode"] == input_mode)].copy()
        for training_mode in available_modes:
            sub_mode = sub_metric.loc[sub_metric["training_mode"] == training_mode].copy()
            for frac in fraction_order:
                frac_sub = sub_mode.loc[np.isclose(sub_mode["train_fraction_total"].astype(float), frac)].copy()
                if not frac_sub.empty:
                    for _, row in frac_sub.iterrows():
                        plot_data.append({"training_mode": training_mode, "fraction": frac, "model": model_name, "mape": float(row["mape"])})
    plot_df = pd.DataFrame(plot_data)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=300, sharey=True)
    split_display = {
        "total split small data": "Total split",
        "per-dataset small data": "Per-dataset split",
        "fine-group small data": "Fine-group split",
    }
    for idx, training_mode in enumerate(available_modes):
        ax = axes[idx]
        mode_df = plot_df[plot_df["training_mode"] == training_mode]
        for spec_key in major_specs:
            model_name, _ = spec_key
            model_df = mode_df[mode_df["model"] == model_name]
            positions = []
            data = []
            mean_x, mean_y = [], []
            for frac in fraction_order:
                vals = model_df[model_df["fraction"] == frac]["mape"].to_numpy(dtype=float)
                finite_vals = vals[np.isfinite(vals)]
                center = fraction_order.index(frac)
                offset = (major_specs.index(spec_key) - 1) * 0.3
                xpos = center + offset
                positions.append(xpos)
                if len(finite_vals) >= 2:
                    data.append(finite_vals)
                else:
                    data.append(np.array([np.nan, np.nan], dtype=float))
                if len(finite_vals) > 0:
                    mean_x.append(xpos)
                    mean_y.append(float(np.nanmean(finite_vals)))
            vp = ax.violinplot(data, positions=positions, widths=0.25, showmeans=False, showmedians=False, showextrema=False, points=100, bw_method=0.3)
            for pc in vp["bodies"]:
                pc.set_facecolor(palette[spec_key])
                pc.set_alpha(0.7)
                pc.set_edgecolor("white")
                pc.set_linewidth(0.5)
            if mean_x:
                ax.plot(mean_x, mean_y, linestyle="--", linewidth=1.2, color=palette[spec_key], alpha=0.8)
                ax.scatter(mean_x, mean_y, s=60, marker=marker_map[spec_key], color=darker_palette[spec_key], edgecolor="black", linewidth=0.5, label=label_map[spec_key] if idx == 0 else None, zorder=5)
        ax.set_xticks(np.arange(len(fraction_order)))
        ax.set_xticklabels([f"{int(round(f*100))}%" for f in fraction_order], fontsize=10)
        ax.set_xlim(-0.6, len(fraction_order) - 0.4)
        ax.set_ylim(10, 30)
        ax.set_title(split_display.get(training_mode, training_mode), fontsize=12)
        ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.35)
        if idx == 0:
            ax.set_ylabel("MAPE %", fontsize=12)
        ax.set_xlabel("Training data fraction", fontsize=12)
    legend_elements = [
        Line2D([0], [0], marker=marker_map[spec_key], color="w", markerfacecolor=palette[spec_key], markersize=8, markeredgecolor="none", label=label_map[spec_key])
        for spec_key in major_specs
    ]
    fig.legend(handles=legend_elements, frameon=False, fontsize=12, loc="upper center", ncol=3, columnspacing=0.8, handlelength=1.2, handletextpad=0.5, bbox_to_anchor=(0.52, 1.08))
    fig.tight_layout()
    #plt.savefig(FIGURE_DIR / "S13-2.tiff", format="tiff", dpi=500, bbox_inches="tight")
    plt.show()


In [ ]:
plot_df = moe_early_cycle_overall_summary_df.loc[
    moe_early_cycle_overall_summary_df.get("plot_group", "major").astype(str).eq("major")
    & (
        ((moe_early_cycle_overall_summary_df["model"].eq("CCV-basic")) & (moe_early_cycle_overall_summary_df["input_mode"].eq("abs")))
        | ((moe_early_cycle_overall_summary_df["model"].eq("CCV-meta-MoE")) & (moe_early_cycle_overall_summary_df["input_mode"].eq("abs")))
        | ((moe_early_cycle_overall_summary_df["model"].eq("CCV-meta-MoLE")) & (moe_early_cycle_overall_summary_df["input_mode"].eq("abs")))
    )
].copy()
if plot_df.empty:
    print("No early-cycle major rows found in metadata-routed early-cycle cache.")
else:
    split_order = ["total split", "per-dataset split", "fine-group split"]
    available_splits = [name for name in split_order if name in plot_df["split_protocol"].astype(str).unique().tolist()]
    major_specs = [("CCV-basic", "abs"), ("CCV-meta-MoE", "abs"), ("CCV-meta-MoLE", "abs")]
    label_map = {
        ("CCV-basic", "abs"): "CCV-basic",
        ("CCV-meta-MoE", "abs"): "CCV-meta-MoE",
        ("CCV-meta-MoLE", "abs"): "CCV-meta-MoLE",
    }
    palette = {
        ("CCV-basic", "abs"): "#E15759",
        ("CCV-meta-MoE", "abs"): "#4E79A7",
        ("CCV-meta-MoLE", "abs"): "#76B7B2",
    }
    marker_map = {
        ("CCV-basic", "abs"): "D",
        ("CCV-meta-MoE", "abs"): "o",
        ("CCV-meta-MoLE", "abs"): "s",
    }
    cycle_order = sorted(plot_df["prefix_end_cycle"].dropna().astype(int).unique().tolist())
    yticks_config = {
        "total split": {"ticks": [10, 15, 20], "ylim": (10, 22)},
        "per-dataset split": {"ticks": [10, 15, 20], "ylim": (10, 22)},
        "fine-group split": {"ticks": [10, 15, 20], "ylim": (10, 22)},
    }
    split_display_names = {
        "total split": "Total split",
        "per-dataset split": "Per-dataset split",
        "fine-group split": "Fine-group split",
    }
    fig, axes = plt.subplots(1, len(available_splits), figsize=(12, 4), dpi=300, squeeze=False)
    for ax, split_name in zip(axes.ravel(), available_splits):
        split_sub = plot_df.loc[plot_df["split_protocol"].eq(split_name)].copy()
        for spec_key in major_specs:
            model_name, input_mode = spec_key
            model_sub = split_sub.loc[
                split_sub["model"].eq(model_name) & split_sub["input_mode"].eq(input_mode)
            ].sort_values("prefix_end_cycle")
            if model_sub.empty:
                continue
            x_vals = model_sub["prefix_end_cycle"].to_numpy(dtype=int)
            y_vals = model_sub["mape"].to_numpy(dtype=float)
            std_vals = model_sub["mape_std"].fillna(0.0).to_numpy(dtype=float)
            ax.plot(x_vals, y_vals, marker=marker_map[spec_key], linewidth=2.0, markersize=6, color=palette[spec_key], label=label_map[spec_key])
            ax.fill_between(x_vals, y_vals - std_vals, y_vals + std_vals, color=palette[spec_key], alpha=0.15)
        config = yticks_config[split_name]
        ax.set_xticks(cycle_order)
        ax.set_yticks(config["ticks"])
        ax.set_ylim(*config["ylim"])
        ax.set_title(split_display_names[split_name], fontsize=13, pad=10)
        ax.set_xlabel("Maximum input cycle", fontsize=11)
        if split_name == available_splits[0]:
            ax.set_ylabel("MAPE %", fontsize=11)
        ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.22)
        ax.set_axisbelow(True)
        for spine in ax.spines.values():
            spine.set_linewidth(0.9)
            spine.set_color("#000000")
        ax.tick_params(axis="both", labelsize=10)
    legend_elements = [
        Line2D([0], [0], color=palette[spec_key], marker=marker_map[spec_key], linewidth=2.0, markersize=6, label=label_map[spec_key])
        for spec_key in major_specs
    ]
    fig.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=3, frameon=False, fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    #plt.savefig(FIGURE_DIR / "S13-3.tiff", format="tiff", dpi=500, bbox_inches="tight")
    plt.show()
